# GENROCKET Specification Alignment Audit

This notebook compares `GENROCKET.csv` against the desired variable distributions in `Genrocket_proposed_var_dists(Research).csv` and the dependency logic in `Data generation specification (1).odt`.

Run the notebook top to bottom. The main outputs are:

- Distribution target match table for variables in the research CSV
- Categorical and numeric profile views of the generated dataset
- Parsed dependency tables from the ODT specification
- Dependency scorecards for treatment assignment, outcome generation, treatment effect logic, and internal consistency rules
- Correlation and conditional-rate diagnostics to identify where variables appear too independent

In [ ]:
from pathlib import Path
import re
import zipfile
from xml.etree import ElementTree as ET

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_colwidth', 220)
pd.set_option('display.width', 160)
plt.style.use('default')

## 1. Configure input files

In [ ]:
DATA_PATH = Path(r'C:/Users/i.Nikesh.Desai/OneDrive - Acentra/Documents/GENROCKET.csv')
RESEARCH_PATH = Path(r'C:/Users/i.Nikesh.Desai/Downloads/Genrocket_proposed_var_dists(Research).csv')
SPEC_PATH = Path(r'C:/Users/i.Nikesh.Desai/Downloads/Data generation specification (1).odt')

for path in [DATA_PATH, RESEARCH_PATH, SPEC_PATH]:
    print(f'{path.name}: exists={path.exists()} size={path.stat().st_size if path.exists() else None}')

In [ ]:
def read_csv_fallback(path, **kwargs):
    for encoding in ['utf-8-sig', 'utf-8', 'cp1252', 'latin1']:
        try:
            return pd.read_csv(path, encoding=encoding, **kwargs), encoding
        except UnicodeDecodeError:
            continue
    return pd.read_csv(path, encoding='latin1', **kwargs), 'latin1'

raw, data_encoding = read_csv_fallback(DATA_PATH)
research, research_encoding = read_csv_fallback(RESEARCH_PATH)

raw = raw.replace({'Null': np.nan, 'NULL': np.nan, 'null': np.nan, '': np.nan})
research.columns = [str(c).strip() for c in research.columns]

print('GENROCKET shape:', raw.shape, 'encoding:', data_encoding)
print('Research shape:', research.shape, 'encoding:', research_encoding)
display(raw.head())
display(research.head())

## 2. Normalize generated data for analysis

In [ ]:
df = raw.copy()

def clean_col(name):
    return re.sub(r'[^a-z0-9]+', '_', str(name).strip().lower()).strip('_')

column_lookup = {clean_col(c): c for c in df.columns}

def col(name):
    return column_lookup.get(clean_col(name), name if name in df.columns else None)

def yn_to_num(s):
    if s is None:
        return pd.Series(dtype='float64')
    x = s.copy()
    if not isinstance(x, pd.Series):
        x = pd.Series(x)
    if pd.api.types.is_numeric_dtype(x):
        return pd.to_numeric(x, errors='coerce')
    return x.astype(str).str.strip().str.upper().map({
        'Y': 1, 'YES': 1, 'TRUE': 1, 'T': 1, '1': 1,
        'N': 0, 'NO': 0, 'FALSE': 0, 'F': 0, '0': 0,
        'NAN': np.nan, 'NONE': np.nan, 'NULL': np.nan
    })

binary_like = []
for c in df.columns:
    vals = set(df[c].dropna().astype(str).str.strip().str.upper().unique())
    if vals and vals.issubset({'Y', 'N', 'YES', 'NO', 'TRUE', 'FALSE', '1', '0'}):
        binary_like.append(c)

for c in binary_like:
    df[c + '__num'] = yn_to_num(df[c])

for c in df.columns:
    if c not in raw.columns:
        continue
    if df[c].dtype == object:
        converted = pd.to_numeric(df[c], errors='coerce')
        if converted.notna().mean() >= 0.90:
            df[c] = converted

print('Binary-like columns:', binary_like)
display(pd.DataFrame({'column': raw.columns, 'dtype': raw.dtypes.astype(str).values, 'missing_rate': raw.isna().mean().values}).head(80))

## 3. Parse the ODT specification

In [ ]:
def parse_odt(path):
    ns = {'table': 'urn:oasis:names:tc:opendocument:xmlns:table:1.0'}
    with zipfile.ZipFile(path) as z:
        root = ET.fromstring(z.read('content.xml'))
    paragraphs = []
    for elem in root.iter():
        if elem.tag.endswith('}p') or elem.tag.endswith('}h'):
            text = ' '.join(''.join(elem.itertext()).split())
            if text:
                paragraphs.append(text)
    tables = []
    for table in root.findall('.//table:table', ns):
        rows = []
        for row in table.findall('table:table-row', ns):
            vals = []
            for cell in row.findall('table:table-cell', ns):
                repeat = int(cell.attrib.get('{urn:oasis:names:tc:opendocument:xmlns:table:1.0}number-columns-repeated', '1'))
                text = ' '.join(''.join(cell.itertext()).split())
                vals.extend([text] * min(repeat, 10))
            if any(vals):
                rows.append(vals)
        tables.append(rows)
    return paragraphs, tables

spec_paragraphs, spec_tables = parse_odt(SPEC_PATH)
print('Paragraphs:', len(spec_paragraphs), 'Tables:', len(spec_tables))
for i, rows in enumerate(spec_tables):
    display(Markdown(f'### ODT Table {i+1}'))
    display(pd.DataFrame(rows))

## 4. Distribution checks against the research CSV

In [ ]:
def parse_pct_range(text):
    if pd.isna(text):
        return None
    s = str(text).replace('–', '-').replace('—', '-')
    m = re.search(r'(\d+(?:\.\d+)?)\s*%?\s*-\s*(\d+(?:\.\d+)?)\s*%', s)
    if m:
        return float(m.group(1))/100, float(m.group(2))/100, 'range'
    m = re.search(r'(\d+(?:\.\d+)?)\s*%', s)
    if m:
        target = float(m.group(1))/100
        tolerance = 0.025 if target < 0.30 else 0.05
        return max(0, target - tolerance), min(1, target + tolerance), 'single_pct_plus_tolerance'
    return None

def parse_numeric_valid_range(text):
    if pd.isna(text):
        return None
    s = str(text).lower()
    if '0 to 1' in s or 'from 0 to 1' in s:
        return 0, 1
    if '0 or greater' in s or 'nonnegative' in s:
        return 0, np.inf
    m = re.search(r'(?:range|approximately)\s*(\d+(?:\.\d+)?)\s*-\s*(\d+(?:\.\d+)?)', s)
    if m:
        return float(m.group(1)), float(m.group(2))
    return None

def distribution_status(observed, low, high):
    if pd.isna(observed):
        return 'MISSING'
    if low <= observed <= high:
        return 'PASS'
    near = max(0.02, 0.15 * (high - low if np.isfinite(high-low) else 0.05))
    if (low - near) <= observed <= (high + near):
        return 'WARN'
    return 'FAIL'

dist_rows = []
for _, r in research.iterrows():
    var = str(r.get('variable', '')).strip()
    if not var or var == 'nan':
        continue
    actual_col = col(var)
    rec = r.get('ChatGPT Reccomendation', np.nan)
    expected = r.get('expected_possible_values_or_range', np.nan)
    note = r.get('Synthetic distribution / correlation QA notes', '')
    row = {'variable': var, 'category': r.get('category', ''), 'present_in_genrocket': actual_col is not None, 'dataset_column': actual_col,
           'target_text': rec, 'expected_values': expected, 'qa_note': note}
    if actual_col is None:
        row.update({'observed': np.nan, 'target_low': np.nan, 'target_high': np.nan, 'status': 'MISSING_COLUMN'})
        dist_rows.append(row)
        continue
    pct_target = parse_pct_range(rec)
    expected_lower = str(expected).lower()
    if pct_target and ('binary' in expected_lower or actual_col in binary_like or var.endswith('_flag') or var in ['dual_eligible', 'intervention_flag']):
        observed = yn_to_num(df[actual_col]).mean()
        low, high, target_type = pct_target
        row.update({'measure': 'yes_rate', 'observed': observed, 'target_low': low, 'target_high': high, 'target_type': target_type,
                    'status': distribution_status(observed, low, high)})
    else:
        rng = parse_numeric_valid_range(expected)
        if pd.api.types.is_numeric_dtype(df[actual_col]) and rng:
            low, high = rng
            invalid = ((df[actual_col] < low) | (df[actual_col] > high)).mean()
            row.update({'measure': 'valid_range_invalid_rate', 'observed': invalid, 'target_low': 0, 'target_high': 0,
                        'status': 'PASS' if invalid == 0 else 'FAIL'})
        elif pd.api.types.is_numeric_dtype(df[actual_col]):
            row.update({'measure': 'numeric_profile', 'observed': df[actual_col].mean(), 'target_low': np.nan, 'target_high': np.nan,
                        'status': 'PROFILE_ONLY'})
        else:
            row.update({'measure': 'categorical_profile', 'observed': df[actual_col].nunique(dropna=True), 'target_low': np.nan, 'target_high': np.nan,
                        'status': 'PROFILE_ONLY'})
    dist_rows.append(row)

distribution_audit = pd.DataFrame(dist_rows)
display(distribution_audit.sort_values(['status', 'category', 'variable']))
display(distribution_audit['status'].value_counts(dropna=False).rename_axis('status').reset_index(name='count'))

In [ ]:
plot_df = distribution_audit[distribution_audit['measure'].eq('yes_rate') & distribution_audit['observed'].notna()].copy()
plot_df = plot_df.sort_values('observed')
fig, ax = plt.subplots(figsize=(10, max(5, 0.28 * len(plot_df))))
y = np.arange(len(plot_df))
ax.barh(y, plot_df['observed'], color='#4c78a8', label='Observed yes rate')
ax.hlines(y, plot_df['target_low'], plot_df['target_high'], color='#f58518', linewidth=4, label='Target range')
ax.set_yticks(y)
ax.set_yticklabels(plot_df['variable'])
ax.set_xlim(0, max(0.75, plot_df[['observed', 'target_high']].max().max() * 1.1))
ax.set_xlabel('Rate')
ax.set_title('Binary Distribution Targets: GENROCKET vs Research CSV')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
numeric_cols = [c for c in raw.columns if pd.api.types.is_numeric_dtype(df[c])]
numeric_profile = df[numeric_cols].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).T
display(numeric_profile)

categorical_cols = [c for c in raw.columns if c not in numeric_cols]
for c in categorical_cols:
    vc = raw[c].value_counts(dropna=False, normalize=True).head(15).rename('share').reset_index().rename(columns={'index': c})
    display(Markdown(f'### {c}'))
    display(vc)

## 5. Dependency and internal logic checks

In [ ]:
def get_series(name, numeric=False, binary=False):
    c = col(name)
    if c is None:
        return None
    if binary:
        return yn_to_num(df[c])
    if numeric:
        return pd.to_numeric(df[c], errors='coerce')
    return df[c]

def weighted_risk_tier(s):
    return s.astype(str).str.strip().str.lower().map({'low': 1, 'medium': 2, 'moderate': 2, 'high': 3, 'very high': 4})

def corr(x, y):
    d = pd.concat([x, y], axis=1).dropna()
    if len(d) < 3 or d.iloc[:,0].nunique() < 2 or d.iloc[:,1].nunique() < 2:
        return np.nan
    return d.iloc[:,0].rank().corr(d.iloc[:,1].rank())

def mean_when(flag_name, target_name, target_numeric=True):
    f = get_series(flag_name, binary=True)
    y = get_series(target_name, numeric=target_numeric, binary=not target_numeric)
    if f is None or y is None:
        return np.nan, np.nan, np.nan
    d = pd.concat([f.rename('flag'), y.rename('target')], axis=1).dropna()
    if d.empty or d['flag'].nunique() < 2:
        return np.nan, np.nan, np.nan
    yes = d.loc[d['flag'].eq(1), 'target'].mean()
    no = d.loc[d['flag'].eq(0), 'target'].mean()
    return yes, no, yes - no

def rate_when(flag_name, target_flag_name):
    return mean_when(flag_name, target_flag_name, target_numeric=False)

def check_status(value, threshold, direction='>='):
    if pd.isna(value):
        return 'MISSING'
    if direction == '>=':
        return 'PASS' if value >= threshold else ('WARN' if value >= threshold * 0.5 else 'FAIL')
    if direction == '<=':
        return 'PASS' if value <= threshold else ('WARN' if value <= threshold * 1.5 else 'FAIL')
    return 'PROFILE'

checks = []
def add_check(area, check, observed, expected, status, detail=''):
    checks.append({'area': area, 'check': check, 'observed': observed, 'expected': expected, 'status': status, 'detail': detail})

# Treatment assignment specification.
treat = get_series('intervention_flag', binary=True)
if treat is not None:
    rate = treat.mean()
    add_check('Treatment assignment', 'Overall intervention rate', rate, 'Approximately 40%', distribution_status(rate, 0.35, 0.45))
    tier = get_series('risk_tier')
    if tier is not None:
        tier_rates = pd.concat([tier.rename('risk_tier'), treat.rename('intervention')], axis=1).dropna().groupby('risk_tier')['intervention'].mean()
        targets = {'Low': .25, 'Moderate': .45, 'Medium': .45, 'High': .65, 'Very High': .65}
        for k, v in tier_rates.items():
            target = targets.get(str(k), np.nan)
            add_check('Treatment assignment', f'Intervention rate by risk_tier={k}', v, f'{target:.0%}' if pd.notna(target) else 'No explicit target',
                      distribution_status(v, max(0, target-.08), min(1, target+.08)) if pd.notna(target) else 'PROFILE')

for feature in ['ed_visits_last_6m', 'admits_last_6m', 'Clinical Score', 'utilization_score', 'SDOH Score', 'behavioral_health_risk_flag', 'chf_flag', 'ckd_flag', 'copd_flag', 'diabetes_flag', 'polypharmacy_flag', 'dual_eligible', 'age']:
    s = get_series(feature, numeric=True) if not feature.endswith('_flag') and feature != 'dual_eligible' else get_series(feature, binary=True)
    if s is not None and treat is not None:
        r = corr(s, treat)
        add_check('Treatment assignment', f'{feature} association with intervention', r, 'Positive Spearman correlation expected', check_status(r, .08), 'Higher complexity should increase treatment probability')

for feature in ['gender', 'language', 'service_region']:
    s = get_series(feature)
    if s is not None and treat is not None:
        spread = pd.concat([s.rename('x'), treat.rename('t')], axis=1).dropna().groupby('x')['t'].mean().pipe(lambda x: x.max() - x.min())
        add_check('Treatment assignment', f'{feature} minimal direct effect', spread, 'Small spread across categories', check_status(spread, .20, '<='))

# Outcome generation specification.
outcome = get_series('outcome_ed_90d', numeric=True)
if outcome is not None:
    rate = outcome.mean()
    add_check('Outcome generation', 'Overall 90-day ED outcome rate', rate, 'Approximately 10%', distribution_status(rate, .07, .13))
    tier = get_series('risk_tier')
    if tier is not None:
        tier_rates = pd.concat([tier.rename('risk_tier'), outcome.rename('outcome')], axis=1).dropna().groupby('risk_tier')['outcome'].mean()
        targets = {'Low': .04, 'Moderate': .10, 'Medium': .10, 'High': .22, 'Very High': .22}
        for k, v in tier_rates.items():
            target = targets.get(str(k), np.nan)
            add_check('Outcome generation', f'90-day ED rate by risk_tier={k}', v, f'{target:.0%}' if pd.notna(target) else 'No explicit target',
                      distribution_status(v, max(0, target-.06), min(1, target+.06)) if pd.notna(target) else 'PROFILE')

for feature in ['ed_visits_last_6m', 'admits_last_6m', 'behavioral_health_risk_flag', 'substance_use_flag', 'housing_instability_flag', 'transportation_barrier_flag', 'food_insecurity_flag', 'chf_flag', 'ckd_flag', 'copd_flag', 'diabetes_flag', 'polypharmacy_flag', 'age']:
    s = get_series(feature, numeric=True) if not feature.endswith('_flag') else get_series(feature, binary=True)
    if s is not None and outcome is not None:
        r = corr(s, outcome)
        add_check('Outcome generation', f'{feature} association with outcome_ed_90d', r, 'Positive association expected', check_status(r, .05))

pdc = get_series('med_adherence_pdc', numeric=True)
if pdc is not None and outcome is not None:
    r = corr(pdc, outcome)
    add_check('Outcome generation', 'med_adherence_pdc association with outcome_ed_90d', r, 'Negative association expected', check_status(-r, .03))

# Core dependency requirements from ODT and research notes.
pairs = [
    ('diabetes_flag', 'ckd_flag', 'Clinical burden'), ('diabetes_flag', 'polypharmacy_flag', 'Clinical burden'),
    ('chf_flag', 'ckd_flag', 'Clinical burden'), ('depression_flag', 'behavioral_health_risk_flag', 'Behavioral health'),
    ('anxiety_flag', 'behavioral_health_risk_flag', 'Behavioral health'), ('substance_use_flag', 'behavioral_health_risk_flag', 'Behavioral health'),
    ('substance_use_flag', 'opioid_flag', 'Behavioral health'), ('housing_instability_flag', 'food_insecurity_flag', 'SDOH'),
    ('housing_instability_flag', 'transportation_barrier_flag', 'SDOH'), ('housing_instability_flag', 'utilities_insecurity_flag', 'SDOH'),
    ('transportation_barrier_flag', 'food_insecurity_flag', 'SDOH'), ('polypharmacy_flag', 'pharmacy_review_flag', 'Post treatment'),
    ('high_cost_drug_flag', 'pharmacy_review_flag', 'Post treatment'), ('food_insecurity_flag', 'community_referral_flag', 'Post treatment'),
    ('housing_instability_flag', 'community_referral_flag', 'Post treatment'), ('transportation_barrier_flag', 'community_referral_flag', 'Post treatment'),
    ('behavioral_health_risk_flag', 'notes_escalation_flag', 'Post treatment')
]
for a, b, area in pairs:
    yes, no, delta = rate_when(a, b)
    add_check(area, f'{a} should increase {b}', delta, 'Positive conditional-rate lift', check_status(delta, .05), f'rate if yes={yes:.3f} vs no={no:.3f}' if pd.notna(delta) else '')

numeric_pairs = [
    ('diabetes_flag', 'rx_count_last_6m', 'Clinical burden'), ('diabetes_flag', 'specialist_visits_last_6m', 'Clinical burden'), ('diabetes_flag', 'admits_last_6m', 'Clinical burden'),
    ('chf_flag', 'ed_visits_last_6m', 'Clinical burden'), ('chf_flag', 'admits_last_6m', 'Clinical burden'), ('chf_flag', 'specialist_visits_last_6m', 'Clinical burden'),
    ('behavioral_health_risk_flag', 'ed_visits_last_6m', 'Behavioral health'), ('behavioral_health_risk_flag', 'admits_last_6m', 'Behavioral health'), ('behavioral_health_risk_flag', 'SDOH Score', 'Behavioral health'),
    ('substance_use_flag', 'ed_visits_last_6m', 'Behavioral health'), ('transportation_barrier_flag', 'ed_visits_last_6m', 'SDOH'), ('transportation_barrier_flag', 'pcp_visits_last_6m', 'SDOH'),
    ('polypharmacy_flag', 'rx_count_last_6m', 'Pharmacy'), ('high_cost_drug_flag', 'total_cost_last_6m', 'Pharmacy'), ('notes_escalation_flag', 'avg_call_duration_min', 'Post treatment')
]
for a, b, area in numeric_pairs:
    yes, no, delta = mean_when(a, b, target_numeric=True)
    expected = 'Positive mean lift' if not (a == 'transportation_barrier_flag' and b == 'pcp_visits_last_6m') else 'Negative mean lift'
    value = -delta if expected.startswith('Negative') else delta
    add_check(area, f'{a} relationship with {b}', delta, expected, check_status(value, .05), f'mean if yes={yes:.3f} vs no={no:.3f}' if pd.notna(delta) else '')

# Interactions and nonlinear requirements.
chf = get_series('chf_flag', binary=True)
ckd = get_series('ckd_flag', binary=True)
ed6 = get_series('ed_visits_last_6m', numeric=True)
if chf is not None and ckd is not None and ed6 is not None:
    both = ((chf == 1) & (ckd == 1)).astype(float)
    one = (((chf == 1) ^ (ckd == 1))).astype(float)
    d = pd.concat([both.rename('both'), one.rename('one'), ed6.rename('ed')], axis=1).dropna()
    both_mean = d.loc[d.both.eq(1), 'ed'].mean()
    one_mean = d.loc[d.one.eq(1), 'ed'].mean()
    add_check('Interactions', 'CHF + CKD should exceed either condition alone for ED utilization', both_mean - one_mean, 'Positive lift for combined condition', check_status(both_mean - one_mean, .10), f'both={both_mean:.3f}, exactly_one={one_mean:.3f}')

if ed6 is not None and outcome is not None:
    buckets = pd.cut(ed6, bins=[-np.inf, 0, 1, 2, np.inf], labels=['0', '1', '2', '3+'])
    ed_curve = pd.concat([buckets.rename('prior_ed_bucket'), outcome.rename('outcome')], axis=1).dropna().groupby('prior_ed_bucket', observed=True)['outcome'].mean()
    display(Markdown('### Outcome rate by prior ED bucket'))
    display(ed_curve.reset_index())
    if len(ed_curve) >= 4:
        jump_after_3 = ed_curve.loc['3+'] - ed_curve.loc['2']
        add_check('Interactions', 'ED outcome should rise more rapidly after 3+ prior ED visits', jump_after_3, 'Positive jump after 3+ visits', check_status(jump_after_3, .05))

# Treatment effect and population segment logic.
te = get_series('true_treatment_effect', numeric=True)
if te is not None:
    for feature in ['transportation_barrier_flag', 'housing_instability_flag', 'food_insecurity_flag', 'behavioral_health_risk_flag', 'depression_flag', 'anxiety_flag']:
        yes, no, delta = mean_when(feature, 'true_treatment_effect', target_numeric=True)
        add_check('Treatment effect', f'{feature} should increase true treatment effect', delta, 'Positive mean lift', check_status(delta, .02), f'mean if yes={yes:.3f} vs no={no:.3f}' if pd.notna(delta) else '')
    if pdc is not None:
        r = corr(pdc, te)
        add_check('Treatment effect', 'Lower medication adherence should increase treatment effect', -r, 'Negative correlation between PDC and treatment effect', check_status(-r, .03), f'corr(PDC, TTE)={r:.3f}' if pd.notna(r) else '')

# Internal consistency checks from research notes.
attempts = get_series('outreach_attempts', numeric=True)
contacts = get_series('successful_contacts', numeric=True)
if attempts is not None and contacts is not None:
    invalid = (contacts > attempts).mean()
    add_check('Internal consistency', 'successful_contacts <= outreach_attempts', invalid, '0 invalid rows', 'PASS' if invalid == 0 else 'FAIL')

avg_call = get_series('avg_call_duration_min', numeric=True)
max_call = get_series('max_call_duration_min', numeric=True)
if avg_call is not None and max_call is not None:
    invalid_order = (avg_call > max_call).mean()
    add_check('Internal consistency', 'avg_call_duration_min <= max_call_duration_min', invalid_order, '0 invalid rows', 'PASS' if invalid_order == 0 else 'FAIL')
    if contacts is not None:
        no_contact_has_call = ((contacts.fillna(0) == 0) & (avg_call.notna() | max_call.notna())).mean()
        add_check('Internal consistency', 'Call durations blank when successful_contacts=0', no_contact_has_call, '0 invalid rows', 'PASS' if no_contact_has_call == 0 else 'FAIL')

dual = get_series('dual_eligible', binary=True)
plan = get_series('plan_type')
if dual is not None and plan is not None:
    dual_plan = plan.astype(str).str.lower().str.contains('dual|medicare', na=False).astype(float)
    agreement = (dual == dual_plan).mean()
    add_check('Internal consistency', 'dual_eligible should align with Dual/Medicare plan_type', agreement, 'High agreement expected', check_status(agreement, .70))

preg = get_series('pregnancy_flag', binary=True)
age = get_series('age', numeric=True)
gender = get_series('gender')
if preg is not None:
    preg_rate = preg.mean()
    add_check('Internal consistency', 'pregnancy_flag prevalence', preg_rate, '3%-6% per research CSV', distribution_status(preg_rate, .03, .06))
    if age is not None:
        invalid_age = ((preg == 1) & ~age.between(12, 55)).mean()
        add_check('Internal consistency', 'pregnancy_flag age constraint', invalid_age, 'No pregnancy outside plausible age range', 'PASS' if invalid_age == 0 else 'FAIL')

checks_df = pd.DataFrame(checks)
display(checks_df.sort_values(['status', 'area', 'check']))
display(checks_df.groupby(['area', 'status']).size().rename('count').reset_index())

## 6. Dependency heatmap and heavy-tail checks

In [ ]:
key_vars = [
    'age', 'dual_eligible', 'diabetes_flag', 'chf_flag', 'ckd_flag', 'copd_flag', 'asthma_flag',
    'depression_flag', 'anxiety_flag', 'substance_use_flag', 'behavioral_health_risk_flag',
    'food_insecurity_flag', 'housing_instability_flag', 'transportation_barrier_flag', 'utilities_insecurity_flag',
    'Clinical Score', 'SDOH Score', 'utilization_score', 'pcp_visits_last_6m', 'specialist_visits_last_6m',
    'ed_visits_last_6m', 'admits_last_6m', 'observation_stays_last_6m', 'total_cost_last_6m',
    'rx_count_last_6m', 'med_adherence_pdc', 'high_cost_drug_flag', 'opioid_flag', 'polypharmacy_flag',
    'intervention_flag', 'outcome_ed_90d', 'true_treatment_effect'
]
corr_data = {}
for v in key_vars:
    c = col(v)
    if c is None:
        continue
    if c in binary_like or v.endswith('_flag') or v in ['dual_eligible', 'intervention_flag']:
        corr_data[v] = yn_to_num(df[c])
    else:
        corr_data[v] = pd.to_numeric(df[c], errors='coerce')
corr_df = pd.DataFrame(corr_data)
spearman = corr_df.rank().corr()

fig, ax = plt.subplots(figsize=(13, 11))
im = ax.imshow(spearman.fillna(0), vmin=-1, vmax=1, cmap='coolwarm')
ax.set_xticks(range(len(spearman.columns)))
ax.set_yticks(range(len(spearman.index)))
ax.set_xticklabels(spearman.columns, rotation=90, fontsize=8)
ax.set_yticklabels(spearman.index, fontsize=8)
ax.set_title('Spearman Correlation Heatmap for Key Dependency Variables')
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

display(spearman.round(3))

In [ ]:
heavy_tail_rows = []
for metric in ['ed_visits_last_6m', 'admits_last_6m', 'observation_stays_last_6m', 'total_cost_last_6m']:
    s = get_series(metric, numeric=True)
    if s is None or s.sum() == 0:
        continue
    threshold = s.quantile(.90)
    top_share = s[s >= threshold].sum() / s.sum()
    heavy_tail_rows.append({'metric': metric, 'p90_threshold': threshold, 'top_10pct_share_of_total': top_share, 'interpretation': 'Higher values indicate stronger heavy-tail behavior'})
display(pd.DataFrame(heavy_tail_rows))

fig, axes = plt.subplots(1, len(heavy_tail_rows), figsize=(4 * max(1, len(heavy_tail_rows)), 3))
if len(heavy_tail_rows) == 1:
    axes = [axes]
for ax, row in zip(axes, heavy_tail_rows):
    s = get_series(row['metric'], numeric=True).dropna().sort_values().reset_index(drop=True)
    cumulative = s.cumsum() / s.sum() if s.sum() else s.cumsum()
    ax.plot(np.linspace(0, 1, len(cumulative)), cumulative)
    ax.plot([0, 1], [0, 1], color='gray', linestyle='--', linewidth=1)
    ax.set_title(row['metric'])
    ax.set_xlabel('Cumulative members')
    ax.set_ylabel('Cumulative total')
plt.tight_layout()
plt.show()

## 7. Export audit tables

This writes reusable CSV summaries next to the notebook.

In [ ]:
preferred_out_dir = Path.cwd() / 'Outputs' / 'genrocket_audit'
fallback_out_dir = Path.cwd() / 'genrocket_audit_outputs'
try:
    preferred_out_dir.mkdir(parents=True, exist_ok=True)
    out_dir = preferred_out_dir
except PermissionError:
    fallback_out_dir.mkdir(parents=True, exist_ok=True)
    out_dir = fallback_out_dir
distribution_audit.to_csv(out_dir / 'distribution_audit.csv', index=False)
checks_df.to_csv(out_dir / 'dependency_checks.csv', index=False)
spearman.to_csv(out_dir / 'key_variable_spearman_correlations.csv')
print('Wrote audit outputs to:', out_dir)